<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_4_vector_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas numpy tqdm

In [3]:
import os
import json
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROJECT_DIR = "/content/drive/MyDrive/llm"

PROCESSED_CORPUS_DIR = f"{PROJECT_DIR}/rag_corpus_processed"
VECTOR_DB_DIR = f"{PROJECT_DIR}/vector_db"

os.makedirs(VECTOR_DB_DIR, exist_ok=True)

FINAL_RAG_CHUNKS_PATH = f"{PROCESSED_CORPUS_DIR}/final_rag_chunks.csv"

FAISS_INDEX_PATH = f"{VECTOR_DB_DIR}/hf_rag_section_faiss.index"
CHUNK_METADATA_PATH = f"{VECTOR_DB_DIR}/section_chunk_metadata.csv"
CHUNK_EMBEDDINGS_PATH = f"{VECTOR_DB_DIR}/section_chunk_embeddings.npy"
VECTOR_CONFIG_PATH = f"{VECTOR_DB_DIR}/section_vector_db_config.json"
RETRIEVAL_TEST_RESULTS_PATH = f"{VECTOR_DB_DIR}/section_retrieval_test_results.csv"

print("Final RAG chunks path:", FINAL_RAG_CHUNKS_PATH)
print("Vector DB directory:", VECTOR_DB_DIR)

Final RAG chunks path: /content/drive/MyDrive/llm/rag_corpus_processed/final_rag_chunks.csv
Vector DB directory: /content/drive/MyDrive/llm/vector_db


In [ ]:
final_rag_chunks = pd.read_csv(FINAL_RAG_CHUNKS_PATH)

required_cols = [
    "chunk_id",
    "document_id",
    "document_name",
    "source_name",
    "document_type",
    "audience",
    "section_title",
    "section_path",
    "page_start",
    "page_end",
    "chunk_text",
    "retrieval_text",
    "include_in_index"
]

missing_cols = [c for c in required_cols if c not in final_rag_chunks.columns]

if missing_cols:
    raise ValueError(f"Missing columns in final_rag_chunks.csv: {missing_cols}")

index_chunks = final_rag_chunks[final_rag_chunks["include_in_index"] == 1].copy()
index_chunks = index_chunks.reset_index(drop=True)

index_chunks["retrieval_text"] = index_chunks["retrieval_text"].fillna("")
index_chunks["chunk_text"] = index_chunks["chunk_text"].fillna("")

if index_chunks.empty:
    raise ValueError("No chunks available for indexing.")

print("Final chunks loaded:", final_rag_chunks.shape)
print("Chunks selected for indexing:", index_chunks.shape)

display(index_chunks.head())

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=embedding_device
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding device:", embedding_device)

In [ ]:
texts_to_embed = index_chunks["retrieval_text"].tolist()

print("Number of chunks to embed:", len(texts_to_embed))

chunk_embeddings = embedding_model.encode(
    texts_to_embed,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

chunk_embeddings = chunk_embeddings.astype("float32")

print("Embedding shape:", chunk_embeddings.shape)
print("Embedding dtype:", chunk_embeddings.dtype)

if chunk_embeddings.shape[0] != len(index_chunks):
    raise ValueError("Embedding row count does not match chunk metadata row count.")

In [7]:
import faiss

embedding_dim = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(chunk_embeddings)

print("FAISS index built.")
print("Embedding dimension:", embedding_dim)
print("Indexed vectors:", faiss_index.ntotal)

if faiss_index.ntotal != len(index_chunks):
    raise ValueError("FAISS vector count does not match metadata row count.")

FAISS index built.
Embedding dimension: 768
Indexed vectors: 969


In [ ]:
faiss.write_index(faiss_index, FAISS_INDEX_PATH)

index_chunks.to_csv(CHUNK_METADATA_PATH, index=False)

np.save(CHUNK_EMBEDDINGS_PATH, chunk_embeddings)

vector_config = {
    "created_at": datetime.now().isoformat(),
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_device": embedding_device,
    "embedding_dimension": int(embedding_dim),
    "faiss_index_type": "IndexFlatIP",
    "normalized_embeddings": True,
    "similarity": "cosine_similarity_via_inner_product",
    "chunk_source": "section_subsection_final_rag_chunks",
    "number_of_chunks_indexed": int(len(index_chunks)),
    "paths": {
        "final_rag_chunks": FINAL_RAG_CHUNKS_PATH,
        "faiss_index": FAISS_INDEX_PATH,
        "chunk_metadata": CHUNK_METADATA_PATH,
        "chunk_embeddings": CHUNK_EMBEDDINGS_PATH
    }
}

with open(VECTOR_CONFIG_PATH, "w") as f:
    json.dump(vector_config, f, indent=2)

print("Saved FAISS index:", FAISS_INDEX_PATH)
print("Saved chunk metadata:", CHUNK_METADATA_PATH)
print("Saved embeddings:", CHUNK_EMBEDDINGS_PATH)
print("Saved vector DB config:", VECTOR_CONFIG_PATH)

In [9]:
def build_query_text(question, patient_context=None):
    if patient_context is None or str(patient_context).strip() == "":
        return str(question).strip()

    return f"""
Clinical question:
{question}

Relevant patient context:
{patient_context}
""".strip()


def embed_query(question, patient_context=None):
    query_text = build_query_text(question, patient_context)

    # BGE retrieval instruction.
    query_text = "Represent this sentence for searching relevant passages: " + query_text

    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return query_embedding.astype("float32")


def search_hf_rag_corpus(question, patient_context=None, top_k=5):
    query_embedding = embed_query(question, patient_context)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx < 0:
            continue

        row = index_chunks.iloc[int(idx)].to_dict()

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "document_name": row["document_name"],
            "source_name": row["source_name"],
            "document_type": row["document_type"],
            "audience": row["audience"],
            "section_title": row["section_title"],
            "section_path": row["section_path"],
            "page_start": row["page_start"],
            "page_end": row["page_end"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(results)

In [ ]:
test_queries = [
    "What follow-up is needed after discharge for acute heart failure?",
    "What should nurses monitor after discharge in a heart failure patient?",
    "What warning signs should a heart failure patient watch for at home?",
    "How is heart failure classified by left ventricular ejection fraction?",
    "What renal function and electrolyte monitoring is needed for ACE inhibitors, ARBs, ARNIs, or MRAs?",
    "What should be monitored during diuretic treatment for congestion?"
]

all_results = []

for query in test_queries:
    print("\n" + "=" * 120)
    print("QUERY:", query)

    results = search_hf_rag_corpus(query, top_k=5)

    display(
        results[
            [
                "rank",
                "score",
                "document_id",
                "document_type",
                "section_title",
                "page_start",
                "page_end",
                "chunk_id"
            ]
        ]
    )

    for _, row in results.iterrows():
        print("\n---")
        print(f"Rank: {row['rank']}")
        print(f"Score: {row['score']:.4f}")
        print(f"Source: {row['document_name']}")
        print(f"Section: {row['section_path']}")
        print(f"Pages: {row['page_start']}-{row['page_end']}")
        print(row["chunk_text"][:1200])

    temp = results.copy()
    temp["query"] = query
    all_results.append(temp)

retrieval_test_results = pd.concat(all_results, ignore_index=True)

retrieval_test_results.to_csv(RETRIEVAL_TEST_RESULTS_PATH, index=False)

print("Saved retrieval test results:", RETRIEVAL_TEST_RESULTS_PATH)

In [ ]:
loaded_index = faiss.read_index(FAISS_INDEX_PATH)
loaded_metadata = pd.read_csv(CHUNK_METADATA_PATH)

print("Loaded FAISS vectors:", loaded_index.ntotal)
print("Loaded metadata rows:", loaded_metadata.shape[0])

if loaded_index.ntotal != loaded_metadata.shape[0]:
    raise ValueError("Reload failed: FAISS vector count and metadata row count do not match.")

print("Reload verification successful.")

In [ ]:
required_outputs = [
    FAISS_INDEX_PATH,
    CHUNK_METADATA_PATH,
    CHUNK_EMBEDDINGS_PATH,
    VECTOR_CONFIG_PATH,
    RETRIEVAL_TEST_RESULTS_PATH
]

for path in required_outputs:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing expected output: {path}")
    print("Found:", path)

print("\nStep 6 complete.")
print("Section/subsection-aware RAG vector database is ready.")